In [50]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [51]:
train = pd.read_csv("/content/fakenews_with_labels.csv", encoding='latin1', on_bad_lines='skip')
test = pd.read_csv("/content/FakeNews_no_labels.csv", encoding='latin1', on_bad_lines='skip')


In [52]:
print(train.columns)
print(train.head())

Index(['ÿtitle', 'text', 'subject', 'date', 'label'], dtype='object')
                                              ÿtitle  \
0  VIDEO: HARLEM BAR Kicks Customers Out For Wear...   
1     PROBLEM: Trump vs. The US Intelligence Machine   
2   Investigators Reveal How Ex-DNC Staffer Likel...   
3  NOT KIDDING: Lawmakers To Decide If Women Can ...   
4  GERMANY: 10,000 Muslims Allegedly Registered T...   

                                                text      subject  \
0  A large group of very diverse young adults who...    left-news   
1  21st Century Wire says The intelligence agenci...  Middle-east   
2  If you pay attention to right-wing media, and ...         News   
3  A Berkeley law that makes public displays of t...    left-news   
4  After allegedly registering 10,000 Muslims to ...     politics   

               date  label  
0       May 6, 2017  False  
1  January 17, 2017  False  
2     June 20, 2017  False  
3       Sep 4, 2017  False  
4      Jun 17, 2017  False  


In [53]:
train['combined'] = (train['ÿtitle'].fillna('') + " " + train['text'].fillna('')).str.strip()
test['combined'] = (test['ÿtitle'].fillna('') + " " + test['text'].fillna('')).str.strip()

In [54]:
train.columns = train.columns.str.replace('ÿ', '')
test.columns = test.columns.str.replace('ÿ', '')

In [55]:
train['label'] = train['label'].astype(str).str.strip().str.upper()

In [56]:
train['label'] = train['label'].map({'TRUE':1, 'FALSE':0})

In [57]:
train = train.dropna(subset=['label'])

In [58]:
print(train['label'].isna().sum())

0


In [59]:
import re

def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

train['combined'] = train['combined'].apply(clean)
test['combined'] = test['combined'].apply(clean)

In [60]:
from sklearn.model_selection import train_test_split

X = train['combined']
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=20000,
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

In [62]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=300, class_weight='balanced')
model.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=300)

In [63]:
y_pred = model.predict(X_val_vec)
accuracy = accuracy_score(y_val, y_pred)
print(" Validation Accuracy:", accuracy)

 Validation Accuracy: 0.9911966987620358


In [64]:
X_full_vec = vectorizer.fit_transform(X)
model.fit(X_full_vec, y)


LogisticRegression(class_weight='balanced', max_iter=300)

In [65]:
test_vec = vectorizer.transform(test['combined'])
test['label'] = model.predict(test_vec)


In [66]:
test['label'] = test['label'].map({1:'TRUE', 0:'FALSE'})

In [67]:
test.to_csv("no_label.csv", index=False)

In [68]:
test.to_csv("no_label.csv", index=False)
print("no_label.csv saved successfully!")

no_label.csv saved successfully!


In [69]:
from google.colab import files
files.download("no_label.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [70]:
from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      1637
         1.0       0.99      0.99      0.99      1998

    accuracy                           0.99      3635
   macro avg       0.99      0.99      0.99      3635
weighted avg       0.99      0.99      0.99      3635

